In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df = pd.read_csv("Human Development Index - Full.csv")

In [8]:
# -----------------------------------------
# CREATE A CLEAN LONG-FORM DATAFRAME
# -----------------------------------------

# 1. Columns that should remain unchanged
id_cols = [
    'ISO3',
    'Country',
    'Human Development Groups',
    'UNDP Developing Regions',
    'HDI Rank (2021)'
]

# 2. Every column after those 5 is a year-based variable
year_cols = df.columns[5:].tolist()

# 3. Melt the year-based columns
#    This preserves every value while putting the year-based
#    column names into one column.
long_df = df[id_cols + year_cols].melt(
    id_vars=id_cols,
    var_name='Indicator_Year',
    value_name='Value'
)

# 4. Separate the indicator name from the year
#    Example:
#    "Human Development Index (1990)"
#    becomes:
#    Indicator = "Human Development Index"
#    Year = 1990
long_df[['Indicator', 'Year']] = long_df['Indicator_Year'].str.extract(
    r'^(.*?) \((\d{4})\)$'
)

# 5. Convert Year from text to an integer
long_df['Year'] = long_df['Year'].astype(int)

# 6. We no longer need the original combined column
long_df = long_df.drop(columns='Indicator_Year')

# 7. Put the columns in the order we want
long_df = long_df[
    id_cols + ['Indicator', 'Year', 'Value']
]

In [10]:
long_df

,ISO3,Country,Human Development Groups,UNDP Developing Regions,HDI Rank (2021),Indicator,Year,Value
0,AFG,Afghanistan,Low,SA,180.0,Human Development Index,1990,0.273
1,AGO,Angola,Medium,SSA,148.0,Human Development Index,1990,NaN
2,ALB,Albania,High,ECA,67.0,Human Development Index,1990,0.647
3,AND,Andorra,Very High,NaN,40.0,Human Development Index,1990,NaN
4,ARE,United Arab Emirates,Very High,AS,26.0,Human Development Index,1990,0.728
...,...,...,...,...,...,...,...,...
170620,WSM,Samoa,High,EAP,111.0,Material footprint per capita (tonnes),2021,NaN
170621,YEM,Yemen,Low,AS,183.0,Material footprint per capita (tonnes),2021,2.890
170622,ZAF,South Africa,High,SSA,109.0,Material footprint per capita (tonnes),2021,7.550
170623,ZMB,Zambia,Medium,SSA,154.0,Material footprint per capita (tonnes),2021,2.380


In [12]:
#long-format data and reshapes it again so that each indicator becomes its own column.
clean_df = long_df.pivot_table(
    index=['ISO3', 'Country', 'Human Development Groups',
           'UNDP Developing Regions', 'HDI Rank (2021)', 'Year'],
    columns='Indicator',
    values='Value',
    aggfunc='first'
).reset_index()

#Remove the column index label
clean_df.columns.name = None

In [15]:
clean_df.head()

,ISO3,Country,Human Development Groups,UNDP Developing Regions,HDI Rank (2021),Year,"Adolescent Birth Rate (births per 1,000 women ages 15-19)",Carbon dioxide emissions per capita (production) (tonnes),Coefficient of human inequality,Difference from HDI rank,...,Life Expectancy at Birth,"Life Expectancy at Birth, female","Life Expectancy at Birth, male",Material footprint per capita (tonnes),"Maternal Mortality Ratio (deaths per 100,000 live births)",Mean Years of Schooling,"Mean Years of Schooling, female","Mean Years of Schooling, male",Overall loss (%),Planetary pressures-adjusted Human Development Index
0,AFG,Afghanistan,Low,SA,180.0,1990,142.960,0.209727,NaN,NaN,...,45.9672,48.3973,43.7088,2.33,1500.0,0.971125,0.342503,1.311020,NaN,0.270
1,AFG,Afghanistan,Low,SA,180.0,1991,147.525,0.182525,NaN,NaN,...,46.6631,49.1439,44.3528,2.28,1530.0,1.019356,0.371860,1.385486,NaN,0.276
2,AFG,Afghanistan,Low,SA,180.0,1992,147.521,0.095233,NaN,NaN,...,47.5955,50.3197,45.0697,2.35,1520.0,1.067586,0.401218,1.459952,NaN,0.284
3,AFG,Afghanistan,Low,SA,180.0,1993,147.896,0.084285,NaN,NaN,...,51.4664,52.7389,50.2164,2.10,1480.0,1.115817,0.430575,1.534418,NaN,0.294
4,AFG,Afghanistan,Low,SA,180.0,1994,155.669,0.075054,NaN,NaN,...,51.4945,53.5442,49.5307,1.83,1540.0,1.164047,0.459933,1.608884,NaN,0.289


In [16]:
#Check for duplicate country-year combinations
clean_df.duplicated(
    subset=['ISO3', 'Country', 'Year']
).sum()

np.int64(0)